**Eight notebooks** cover the workflow sequence:
1. [create_datasets_yf.ipynb](01_create_datasets_yf.ipynb) : creates datasets using yfinance
2. `sentiment_feature_building`(this notebook): (optional) we calculate the sentiment using finbert on financial news
3. [feature_engineering_yf](03_feature_engineering_yf.ipynb): we compute features from data to later feed into the model
4. [optimizing_xgboost](04_optimizing_xgboost.ipynb): we train a xgboost model to predict returns
5. [evaluate_xgboost](05_evaluate_xgboost.ipynb): we compare the cross-validation performance using various metrics to select the best model. 
6. [model_interpretation](06_model_interpretation.ipynb): we take a closer look at the drivers behind the best model's predictions.
7. [making_out_of_sample_predictions](07_making_out_of_sample_predictions.ipynb): we predict returns for our out-of-sample period
8. [backtest_backtrader](08_backtest_backtrader.ipynb): evaluate the historical performance of our strategy based on our predictive signals

In [1]:
# Import necessary libraries
import pandas as pd
from datetime import datetime, timedelta
from alpaca_trade_api.rest import REST
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import os,sys
import time





/home/sabateri/anaconda3/envs/stock-sentiment/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/sabateri/anaconda3/envs/stock-sentiment/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --------- CONFIG ---------
ALPACA_API_KEY = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")
#APCA_API_SECRET_KEY=ALPACA_SECRET_KEY
BASE_URL = "https://paper-api.alpaca.markets"

#TICKERS = ['AAPL', 'MSFT', 'TSLA']
START_DATE = "2024-01-01"
END_DATE = "2024-05-28"
MAX_ARTICLES_PER_TICKER = 100

In [4]:
# --------- SETUP ---------
api = REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, BASE_URL)

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")
model.eval()

def get_sentiment_scores(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    probs_np = probs.squeeze().numpy()
    return {
        'sent_pos': probs_np[0],
        'sent_neu': probs_np[1],
        'sent_neg': probs_np[2]
    }

def get_sentiment_scores_batch(texts, batch_size=32):
    """Process multiple texts at once for better GPU utilization"""
    all_scores = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)
        
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        # Convert to list of dictionaries
        for prob_array in probs.numpy():
            all_scores.append({
                'sent_pos': prob_array[0],
                'sent_neu': prob_array[1], 
                'sent_neg': prob_array[2]
            })
    
    return all_scores


Get the NASDAQ tickers using the fetcher class we defined

In [ ]:
# scripts_dir = os.path.abspath(os.path.join(os.getcwd(), '../scripts/'))
# sys.path.append(scripts_dir)
# from data_fetcher import StockDataFetcher

# fetcher = StockDataFetcher()
# nasdaq_tickers = fetcher.get_nasdaq_tickers(json_path = '../data/nasdaq_tickers.json')

INFO:data_fetcher:Retrieved 3825 NASDAQ tickers


### Only keep the top most traded stocks, otherwise too computationally expensive

In [ ]:
DATA_FILE = '../data/data_features.h5'
data = pd.read_hdf(DATA_FILE, 'model_data').dropna().sort_index()
data = data[data['dollar_vol_rank'] <= 100]
nasdaq_tickers = data.index.get_level_values(level=0).unique().to_list()

In [18]:
# --------- FETCH & SCORE NEWS ---------
sentiment_records = []

failed_tickers = []

for ticker in nasdaq_tickers:
    ticker_articles = 0
    print(f"Fetching news for {ticker}...")
    try:
        news_items = api.get_news(
            symbol=ticker,
            start=START_DATE,
            end=END_DATE,
            limit=MAX_ARTICLES_PER_TICKER
        )
    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")
        failed_tickers.append(ticker)
        continue

    for item in news_items:
        text = item.headline
        if not text.strip():
            continue

        try:
            #scores = get_sentiment_scores(text)
            scores = get_sentiment_scores_batch(text)
        except Exception as e:
            print(f"Skipping due to FinBERT error: {e}")
            continue

        sentiment_records.append({
            'date': item.created_at.date(),
            'ticker': ticker,
            **scores
        })

        # save progress periodically
        if len(sentiment_records) % 100 == 0:
            pd.DataFrame(sentiment_records).to_csv('../data/sentiment/temp_sentiment_progress.csv', index=False)

        time.sleep(0.1)  # avoid rate-limiting

Fetching news for AAL...


TypeError: 'list' object is not a mapping

In [19]:
import re
def preprocess_text(text):
    # Clean and validate text before sentiment analysis
    text = text.strip()
    #if len(text) < 10:  # Skip very short headlines
    #    return None
    # Remove excessive whitespace, special characters if needed
    text = re.sub(r'\s+', ' ', text)
    return text

In [21]:
# --------- FETCH & SCORE NEWS ---------
sentiment_records = []

failed_tickers = []



for ticker in nasdaq_tickers:
    print(f"Fetching news for {ticker}...")
    try:
        news_items = api.get_news(
            symbol=ticker,
            start=START_DATE,
            end=END_DATE,
            limit=MAX_ARTICLES_PER_TICKER
        )
    except Exception as e:
        print(f"Error fetching news for {ticker}: {e}")
        failed_tickers.append(ticker)
        continue

    # Collect texts for batch processing
    texts_to_process = []
    text_metadata = []

    for item in news_items:
        
        #text = item.headline
        text = preprocess_text(item.headline)
        if not text.strip():
            continue
        
        texts_to_process.append(text)
        text_metadata.append({
            'date': item.created_at.date(),
            'ticker': ticker
        })
        #print(text,item.created_at.date(),ticker)

    # Process all texts for this ticker in batches
    if texts_to_process:
        try:
            batch_scores = get_sentiment_scores_batch(texts_to_process)
            
            # Combine scores with metadata
            for scores, metadata in zip(batch_scores, text_metadata):
                sentiment_records.append({
                    **metadata,
                    **scores
                })
                
        except Exception as e:
            print(f"Skipping ticker {ticker} due to batch processing error: {e}")
            continue



        # save progress periodically
        if len(sentiment_records) % 100 == 0:
            pd.DataFrame(sentiment_records).to_csv('../data/sentiment/temp_sentiment_progress.csv', index=False)

        time.sleep(0.1)  # avoid rate-limiting

Fetching news for AAL...
Fetching news for AAOI...
Fetching news for AAPL...
Fetching news for ACGL...
Fetching news for ACHC...
Fetching news for ADBE...
Fetching news for ADI...
Fetching news for ADP...
Fetching news for ADSK...
Fetching news for ADTN...
Fetching news for AEP...
Fetching news for AGIO...
Fetching news for AGNC...
Fetching news for AKAM...
Fetching news for ALGN...
Fetching news for ALKS...
Fetching news for ALNY...
Fetching news for AMAT...
Fetching news for AMBA...
Fetching news for AMCX...
Fetching news for AMD...
Fetching news for AMED...
Fetching news for AMGN...
Fetching news for AMSC...
Fetching news for AMZN...
Fetching news for ANSS...
Fetching news for APA...
Fetching news for APLS...
Fetching news for APPN...
Fetching news for APPS...
Fetching news for ARGX...
Fetching news for ARWR...
Fetching news for ASML...
Fetching news for AUPH...
Fetching news for AVGO...
Fetching news for AXON...
Fetching news for AZN...
Fetching news for BCRX...
Fetching news for B

In [ ]:
# --------- AGGREGATE ---------
df_sentiment = pd.DataFrame(sentiment_records)

if df_sentiment.empty:
    print("No sentiment data collected.")
else:
    df_daily = (
        df_sentiment
        .groupby(['date', 'ticker'])
        .agg({
        'sent_pos': ['mean', 'std', 'count'],
        'sent_neu': ['mean', 'std'],
        'sent_neg': ['mean', 'std']
        })
        .reset_index()
    )

    # compute extra features
    df_daily['sent_pos_ratio'] = df_daily['sent_pos'] / (df_daily['sent_pos'] + df_daily['sent_neg'] + 1e-8)
    df_daily['sentiment_score'] = df_daily['sent_pos'] - df_daily['sent_neg']
    # confidence metrics
    df_daily['sentiment_confidence'] = 1 - df_daily[('sent_neu', 'mean')]
    df_daily['sentiment_consensus'] = 1 / (1 + df_daily[('sent_pos', 'std')])

    # save to disk
    output_path = "../data/sentiment/sentiment_scores.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_daily.to_csv(output_path, index=False)

    print(f"Saved sentiment features to {output_path}")

Saved sentiment features to ../data/sentiment/sentiment_scores.csv
